<a href="https://colab.research.google.com/github/ksehar99/EyePACS-DL-Blockchain/blob/main/DR_Private_Blockchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# NVM install + Node 22 + Python libraries
!wget -qO- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash > /dev/null 2>&1
!pip install web3 faker -q

print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 52.0 MB/s eta 0:00:00
Done!


In [2]:
import os
os.makedirs("/content/hardhat-project", exist_ok=True)
os.chdir("/content/hardhat-project")

# Purane config files delete karo
!rm -f hardhat.config.cjs hardhat.config.js hardhat.config.ts

# package.json
with open("package.json", "w") as f:
    f.write('{"name":"dr-blockchain","version":"1.0.0"}')

# hardhat.config.js — clean, no comments inside object
with open("hardhat.config.js", "w") as f:
    f.write('module.exports = {\n  solidity: "0.8.22",\n  networks: { hardhat: { chainId: 1337 } }\n};\n')

# Confirm file content
!cat hardhat.config.js

# Node 22 + Hardhat install
!bash -c "source /root/.nvm/nvm.sh && nvm install 22 && nvm use 22 && npm install --save-dev hardhat@2.22.17 --legacy-peer-deps" 2>&1 | tail -3

!bash -c "source /root/.nvm/nvm.sh && nvm use 22 && ./node_modules/.bin/hardhat --version"

module.exports = {
  solidity: "0.8.22",
  networks: { hardhat: { chainId: 1337 } }
};
  npm audit fix --force

Run `npm audit` for details.
Now using node v22.23.1 (npm v10.9.8)
2.22.17


In [3]:
import subprocess, time, os
from web3 import Web3

os.chdir("/content/hardhat-project")

process = subprocess.Popen(
    ["bash", "-c", "source /root/.nvm/nvm.sh && nvm use 22 && ./node_modules/.bin/hardhat node"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("Network start ho raha hai... (10 sec)")
time.sleep(10)

w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))
if w3.is_connected():
    print(f"Private network chal raha hai! Chain ID: {w3.eth.chain_id}")
    print(f"Total accounts: {len(w3.eth.accounts)}")
    for i, acc in enumerate(w3.eth.accounts[:3]):
        bal = w3.from_wei(w3.eth.get_balance(acc), 'ether')
        print(f"  [{i}] {acc} — {bal} ETH")
else:
    _, err = process.communicate(timeout=2)
    print("Error:", err)

Network start ho raha hai... (10 sec)
Private network chal raha hai! Chain ID: 1337
Total accounts: 20
  [0] 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 — 10000 ETH
  [1] 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 — 10000 ETH
  [2] 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC — 10000 ETH


In [4]:
import json
from web3 import Web3

# ── ABI ────────────────────────────────────────────────────────────────
ABI = json.loads('''[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[],"name":"ConsentAlreadyGiven","type":"error"},{"inputs":[],"name":"ConsentNotGiven","type":"error"},{"inputs":[],"name":"NoDiagnosisFound","type":"error"},{"inputs":[],"name":"NotAuthorized","type":"error"},{"inputs":[],"name":"NotOwner","type":"error"},{"inputs":[],"name":"PatientAlreadyExists","type":"error"},{"inputs":[],"name":"PatientNotFound","type":"error"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"ConsentGiven","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"ConsentRevoked","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"address","name":"doctorId","type":"address"},{"indexed":false,"internalType":"bool","name":"diagnosisResult","type":"bool"},{"indexed":false,"internalType":"uint256","name":"confidenceScore","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"DiagnosisUploaded","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"bool","name":"doctorResult","type":"bool"},{"indexed":false,"internalType":"bool","name":"agreedWithAI","type":"bool"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"DoctorDecisionRecorded","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"address","name":"oldDoctor","type":"address"},{"indexed":false,"internalType":"address","name":"newDoctor","type":"address"}],"name":"DoctorReassigned","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"address","name":"admin","type":"address"},{"indexed":false,"internalType":"string","name":"reason","type":"string"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"EmergencyAccessLog","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"address","name":"doctorAddress","type":"address"}],"name":"PatientRegistered","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":false,"internalType":"address","name":"secondDoctor","type":"address"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"SecondOpinionRequested","type":"event"},{"inputs":[{"internalType":"address","name":"doctor","type":"address"}],"name":"authorizeExternalDoctor","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"checkConsent","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"string","name":"reason","type":"string"}],"name":"emergencyAccess","outputs":[{"components":[{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"bool","name":"diagnosisResult","type":"bool"},{"internalType":"uint256","name":"confidenceScore","type":"uint256"},{"internalType":"bytes32","name":"modelVersionHash","type":"bytes32"},{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"},{"internalType":"bool","name":"reviewedByDoctor","type":"bool"}],"internalType":"struct DRDiagnosisResults.Diagnosis[]","name":"","type":"tuple[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"getPatientAddress","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"giveConsent","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"isPatientRegistered","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"isReviewed","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"address","name":"_newDoctor","type":"address"}],"name":"reassignDoctor","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bool","name":"doctorResult","type":"bool"},{"internalType":"bool","name":"agreedWithAI","type":"bool"},{"internalType":"string","name":"notes","type":"string"}],"name":"recordDoctorDecision","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"address","name":"_doctorAddress","type":"address"},{"internalType":"address","name":"_patientAddress","type":"address"}],"name":"registerPatient","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"secondDoctor","type":"address"}],"name":"requestSecondOpinion","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"}],"name":"revokeConsent","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"bool","name":"diagnosisResult","type":"bool"},{"internalType":"uint256","name":"confidenceScore","type":"uint256"},{"internalType":"bytes32","name":"modelVersionHash","type":"bytes32"}],"name":"uploadDiagnosis","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bytes32","name":"imageHash","type":"bytes32"}],"name":"verifyDiagnosis","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"}],"name":"viewDoctorDecisions","outputs":[{"components":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bool","name":"doctorResult","type":"bool"},{"internalType":"bool","name":"agreedWithAI","type":"bool"},{"internalType":"string","name":"notes","type":"string"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"}],"internalType":"struct DRDiagnosisResults.DoctorDecision[]","name":"","type":"tuple[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"}],"name":"viewMyRecords","outputs":[{"components":[{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"bool","name":"diagnosisResult","type":"bool"},{"internalType":"uint256","name":"confidenceScore","type":"uint256"},{"internalType":"bytes32","name":"modelVersionHash","type":"bytes32"},{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"},{"internalType":"bool","name":"reviewedByDoctor","type":"bool"}],"internalType":"struct DRDiagnosisResults.Diagnosis[]","name":"","type":"tuple[]"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"viewPatients","outputs":[{"internalType":"uint256[]","name":"","type":"uint256[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"}],"name":"viewRecords","outputs":[{"components":[{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"bool","name":"diagnosisResult","type":"bool"},{"internalType":"uint256","name":"confidenceScore","type":"uint256"},{"internalType":"bytes32","name":"modelVersionHash","type":"bytes32"},{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"},{"internalType":"bool","name":"reviewedByDoctor","type":"bool"}],"internalType":"struct DRDiagnosisResults.Diagnosis[]","name":"","type":"tuple[]"}],"stateMutability":"view","type":"function"}]''')

# ── Bytecode ────────────────────────────────────────────────────────────
BYTECODE = "..."

# ── Connect to private network ──────────────────────────────────────────
w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))

if not w3.is_connected():
    print("Network se connect nahi ho saka — pehle Cell 4 (network start) chalao")
else:
    print(f"Connected! Chain ID: {w3.eth.chain_id}")

    # Accounts assign karo
    ADMIN   = w3.eth.accounts[0]   # Hospital Admin
    DOCTOR1 = w3.eth.accounts[1]   # Doctor 1
    DOCTOR2 = w3.eth.accounts[2]   # Doctor 2
    PATIENT1 = w3.eth.accounts[3]  # Patient 1
    PATIENT2 = w3.eth.accounts[4]  # Patient 2

    print(f"\nAdmin:   {ADMIN}")
    print(f"Doctor1: {DOCTOR1}")
    print(f"Doctor2: {DOCTOR2}")
    print(f"Patient1: {PATIENT1}")
    print(f"Patient2: {PATIENT2}")

    # Contract deploy karo
    Contract = w3.eth.contract(abi=ABI, bytecode=BYTECODE)
    tx_hash = Contract.constructor().transact({'from': ADMIN})
    tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    CONTRACT_ADDRESS = tx_receipt.contractAddress
    print(f"\nContract deployed at: {CONTRACT_ADDRESS}")

    # Contract instance banao
    contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)
    print("Contract ready!")

Connected! Chain ID: 1337

Admin:   0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Doctor1: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Doctor2: 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC
Patient1: 0x90F79bf6EB2c4f870365E785982E1f101E93b906
Patient2: 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65

Contract deployed at: 0x5FbDB2315678afecb367f032d93F642f64180aa3
Contract ready!


In [5]:
import hashlib

# ── Helpers ────────────────────────────────────────────────────────────
def to_bytes32(hex_str):
    """String hash ko bytes32 mein convert karo"""
    return bytes.fromhex(hex_str)

def sha256_file(path):
    """File ka SHA-256 hash nikalo"""
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

def sha256_string(s):
    """String ka SHA-256 hash nikalo"""
    return hashlib.sha256(s.encode()).hexdigest()

# ── Step 1: Patient register karo (Admin call karta hai) ───────────────
patient_id = 101

tx = contract.functions.registerPatient(
    patient_id,
    DOCTOR1,
    PATIENT1
).transact({'from': ADMIN})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Patient {patient_id} registered!")

# ── Step 2: Patient consent deta hai (Patient khud call karta hai) ─────
tx = contract.functions.giveConsent(
    patient_id
).transact({'from': PATIENT1})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Consent given by Patient {patient_id}!")

# ── Step 3: Consent verify karo ────────────────────────────────────────
consent = contract.functions.checkConsent(patient_id).call()
print(f"Consent status: {consent}")

Patient 101 registered!
Consent given by Patient 101!
Consent status: True


In [6]:
import json
import os
import hashlib
import random
from web3 import Web3

# ── Setup ──────────────────────────────────────────────────────────────
USED_ACCOUNTS_FILE = "/content/drive/MyDrive/EyePACS-DL-Blockchain/used_accounts.json"
USED_IMAGES_FILE   = "/content/drive/MyDrive/EyePACS-DL-Blockchain/used_images.json"
TEST_IMAGES_DIR    = "/content/drive/MyDrive/EyePACS/EyePACS/test/test_merge"
MODEL_VERSION      = "dr_detection_mobilenetv2_v2_threshold_0.30"

# Fixed role assignments
ADMIN_IDX   = 0
DOCTOR_IDXS = [1, 2, 3]
PATIENT_POOL = list(range(4, 20))

accounts = w3.eth.accounts
MODEL_VERSION_HASH = bytes.fromhex(
    hashlib.sha256(MODEL_VERSION.encode()).hexdigest()
)

# ── Helpers ────────────────────────────────────────────────────────────
def load_json(path, default):
    try:
        with open(path) as f:
            return json.load(f)
    except:
        return default

def save_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def get_next_patient_account():
    used = load_json(USED_ACCOUNTS_FILE, [])
    for idx in PATIENT_POOL:
        if idx not in used:
            return idx
    return None

def mark_account_used(idx):
    used = load_json(USED_ACCOUNTS_FILE, [])
    if idx not in used:
        used.append(idx)
    save_json(USED_ACCOUNTS_FILE, used)

def get_random_image():
    used = load_json(USED_IMAGES_FILE, [])
    all_images = [
        f for f in os.listdir(TEST_IMAGES_DIR)
        if f.endswith('.jpeg') and f not in used
    ]
    if not all_images:
        return None, None
    chosen = random.choice(all_images)
    full_path = os.path.join(TEST_IMAGES_DIR, chosen)
    used.append(chosen)
    save_json(USED_IMAGES_FILE, used)
    return full_path, chosen

def image_sha256(path):
    with open(path, 'rb') as f:
        return bytes.fromhex(hashlib.sha256(f.read()).hexdigest())

def clear():
    print("\n" + "="*50 + "\n")

def identify_account(idx):
    if idx == ADMIN_IDX:
        return "admin"
    elif idx in DOCTOR_IDXS:
        return "doctor"
    else:
        return "patient"

# ── Simulate model prediction (DL cell se replace hoga) ───────────────
def run_model(image_path):
    """
    Placeholder — DL model cell se replace hoga.
    Returns: (diagnosis_result: bool, confidence: int)
    """
    # DL integration ke baad yahan actual model.predict() aayega
    import random
    result = random.choice([True, False])
    confidence = random.randint(60, 99)
    return result, confidence

# ══════════════════════════════════════════════════════════════════════
# ADMIN MENU
# ══════════════════════════════════════════════════════════════════════
def admin_menu():
    while True:
        clear()
        print("ADMIN PANEL")
        print("-" * 30)
        print("[1] Register new patient")
        print("[2] Reassign doctor")
        print("[3] Emergency access")
        print("[4] Authorize external doctor")
        print("[5] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            # Next available account
            acc_idx = get_next_patient_account()
            if acc_idx is None:
                print("No available patient accounts left.")
                input("Press Enter to continue...")
                continue

            patient_address = accounts[acc_idx]

            # Patient ID
            try:
                patient_id = int(input("Enter Patient ID: ").strip())
            except:
                print("Invalid ID.")
                input("Press Enter...")
                continue

            # Doctor selection
            print("\nAvailable doctors:")
            for i, d in enumerate(DOCTOR_IDXS):
                print(f"  [{i+1}] Doctor {i+1} ({accounts[d]})")
            try:
                d_choice = int(input("Select doctor (1/2/3): ").strip()) - 1
                doctor_address = accounts[DOCTOR_IDXS[d_choice]]
            except:
                print("Invalid selection.")
                input("Press Enter...")
                continue

            # On-chain register
            try:
                tx = contract.functions.registerPatient(
                    patient_id,
                    doctor_address,
                    patient_address
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)
                mark_account_used(acc_idx)

                print(f"\nPatient registered successfully!")
                print(f"Patient ID : {patient_id}")
                print(f"Account    : [{acc_idx}] {patient_address}")
                print(f"Doctor     : Doctor {d_choice+1}")
                print(f"\nTell patient: Login with account number {acc_idx}")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "2":
            try:
                patient_id = int(input("Enter Patient ID: ").strip())
                print("\nAvailable doctors:")
                for i, d in enumerate(DOCTOR_IDXS):
                    print(f"  [{i+1}] Doctor {i+1} ({accounts[d]})")
                d_choice = int(input("Select new doctor (1/2/3): ").strip()) - 1
                new_doctor = accounts[DOCTOR_IDXS[d_choice]]

                tx = contract.functions.reassignDoctor(
                    patient_id, new_doctor
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)
                print(f"Doctor reassigned successfully!")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "3":
            try:
                patient_id = int(input("Enter Patient ID: ").strip())
                reason = input("Enter reason for emergency access: ").strip()

                records = contract.functions.emergencyAccess(
                    patient_id, reason
                ).call({'from': accounts[ADMIN_IDX]})

                # Emergency access transaction (logged on-chain)
                tx = contract.functions.emergencyAccess(
                    patient_id, reason
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)

                print(f"\nEmergency access granted — logged on-chain.")
                print(f"Total diagnoses: {len(records)}")
                for i, r in enumerate(records):
                    print(f"\n  Diagnosis {i+1}:")
                    print(f"    Result    : {'DR Detected' if r[1] else 'No DR'}")
                    print(f"    Confidence: {r[2]}%")
                    print(f"    Reviewed  : {'Yes' if r[7] else 'Pending'}")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "4":
            try:
                doctor_addr = input("Enter external doctor address (0x...): ").strip()
                tx = contract.functions.authorizeExternalDoctor(
                    doctor_addr
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)
                print("External doctor authorized.")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "5":
            break

# ══════════════════════════════════════════════════════════════════════
# DOCTOR MENU
# ══════════════════════════════════════════════════════════════════════
def doctor_menu(doctor_idx):
    doctor_address = accounts[doctor_idx]
    doctor_num = DOCTOR_IDXS.index(doctor_idx) + 1

    while True:
        clear()
        print(f"DOCTOR {doctor_num} PANEL")
        print(f"Address: {doctor_address}")
        print("-" * 30)
        print("[1] My patients")
        print("[2] Pending reviews")
        print("[3] Upload diagnosis + Record decision")
        print("[4] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            patients = contract.functions.viewPatients().call(
                {'from': doctor_address}
            )
            print(f"\nAssigned patients: {patients if patients else 'None'}")
            input("\nPress Enter to continue...")

        elif choice == "2":
            patients = contract.functions.viewPatients().call(
                {'from': doctor_address}
            )
            pending = []
            for pid in patients:
                reviewed = contract.functions.isReviewed(pid).call()
                if not reviewed:
                    pending.append(pid)

            if pending:
                print(f"\nPending reviews: {pending}")
            else:
                print("\nNo pending reviews.")
            input("\nPress Enter to continue...")

        elif choice == "3":
            try:
                patient_id = int(input("Enter Patient ID: ").strip())

                # Image uthao
                image_path, image_name = get_random_image()
                if image_path is None:
                    print("No unused test images available.")
                    input("Press Enter...")
                    continue

                print(f"\nImage selected: {image_name}")
                print("Running model...")

                # Model predict karo
                diagnosis_result, confidence = run_model(image_path)

                print(f"\nAI Prediction:")
                print(f"  Result    : {'DR Detected' if diagnosis_result else 'No DR'}")
                print(f"  Confidence: {confidence}%")

                # Doctor decision
                print("\nYour decision:")
                print("[a] Agree with AI")
                print("[o] Override AI")
                d_input = input("Select (a/o): ").strip().lower()

                agreed = d_input == 'a'
                if agreed:
                    doctor_result = diagnosis_result
                else:
                    print("[1] DR Detected")
                    print("[2] No DR")
                    r = input("Your diagnosis: ").strip()
                    doctor_result = r == "1"

                notes = input("Notes (optional, press Enter to skip): ").strip()

                # Image hash
                image_hash = image_sha256(image_path)

                # Upload diagnosis on-chain
                tx1 = contract.functions.uploadDiagnosis(
                    patient_id,
                    image_hash,
                    diagnosis_result,
                    confidence,
                    MODEL_VERSION_HASH
                ).transact({'from': doctor_address})
                w3.eth.wait_for_transaction_receipt(tx1)

                # Doctor decision on-chain
                tx2 = contract.functions.recordDoctorDecision(
                    patient_id,
                    doctor_result,
                    agreed,
                    notes
                ).transact({'from': doctor_address})
                w3.eth.wait_for_transaction_receipt(tx2)

                print(f"\nDiagnosis uploaded and decision recorded on-chain!")
                print(f"Image hash: {image_hash.hex()[:20]}...")

                # Tamper verify
                verified = contract.functions.verifyDiagnosis(
                    patient_id, image_hash
                ).call()
                print(f"Tamper verification: {'PASSED' if verified else 'FAILED'}")

            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "4":
            break

# ══════════════════════════════════════════════════════════════════════
# PATIENT MENU
# ══════════════════════════════════════════════════════════════════════
def patient_menu(patient_address):
    # Patient ID dhundo
    patient_id = None
    # Events se dhundenge
    events = contract.events.PatientRegistered.get_logs(from_block=0)
    for e in events:
        pid = e['args']['patientId']
        addr = contract.functions.getPatientAddress(pid).call()
        if addr.lower() == patient_address.lower():
            patient_id = pid
            break

    if patient_id is None:
        print("Patient ID nahi mila. Admin se contact karo.")
        input("Press Enter...")
        return

    while True:
        clear()
        print(f"PATIENT PANEL")
        print(f"Patient ID: {patient_id}")
        print(f"Address   : {patient_address}")
        print("-" * 30)
        print("[1] View my records")
        print("[2] View doctor decisions")
        print("[3] Consent status")
        print("[4] Give consent (cross-hospital data sharing)")
        print("[5] Revoke consent")
        print("[6] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            records = contract.functions.viewMyRecords(
                patient_id
            ).call({'from': patient_address})

            if not records:
                print("\nKoi diagnosis record nahi hai abhi.")
            else:
                print(f"\nTotal diagnoses: {len(records)}")
                for i, r in enumerate(records):
                    print(f"\n  Diagnosis {i+1}:")
                    print(f"    Result    : {'DR Detected' if r[1] else 'No DR'}")
                    print(f"    Confidence: {r[2]}%")
                    print(f"    Date      : Block timestamp {r[6]}")
                    print(f"    Reviewed  : {'Yes' if r[7] else 'Pending doctor review'}")
            input("\nPress Enter to continue...")

        elif choice == "2":
            try:
                decisions = contract.functions.viewDoctorDecisions(
                    patient_id
                ).call({'from': patient_address})

                if not decisions:
                    print("\nKoi doctor decision nahi hai abhi.")
                else:
                    for i, d in enumerate(decisions):
                        print(f"\n  Decision {i+1}:")
                        print(f"    Doctor result : {'DR' if d[1] else 'No DR'}")
                        print(f"    Agreed with AI: {'Yes' if d[2] else 'No (Override)'}")
                        print(f"    Notes         : {d[3] if d[3] else 'None'}")
            except Exception as e:
                print(f"Access denied: {e}")
            input("\nPress Enter to continue...")

        elif choice == "3":
            consent = contract.functions.checkConsent(patient_id).call()
            print(f"\nConsent status: {'Given' if consent else 'Not given'}")
            input("\nPress Enter to continue...")

        elif choice == "4":
            try:
                tx = contract.functions.giveConsent(
                    patient_id
                ).transact({'from': patient_address})
                w3.eth.wait_for_transaction_receipt(tx)
                print("Consent given successfully.")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "5":
            try:
                tx = contract.functions.revokeConsent(
                    patient_id
                ).transact({'from': patient_address})
                w3.eth.wait_for_transaction_receipt(tx)
                print("Consent revoked.")
            except Exception as e:
                print(f"Error: {e}")
            input("\nPress Enter to continue...")

        elif choice == "6":
            break

# ══════════════════════════════════════════════════════════════════════
# MAIN LOGIN
# ══════════════════════════════════════════════════════════════════════
def main():
    while True:
        clear()
        print("DR DIAGNOSIS BLOCKCHAIN SYSTEM")
        print("Private Network | Chain ID 1337")
        print("=" * 50)
        print("\nAvailable accounts:")
        print(f"  [0]      Admin")
        print(f"  [1][2][3] Doctors")
        print(f"  [4-19]   Patients")
        print()

        try:
            idx = int(input("Enter your account number (0-19): ").strip())
            if idx < 0 or idx > 19:
                raise ValueError
        except:
            print("Invalid input.")
            input("Press Enter...")
            continue

        role = identify_account(idx)
        address = accounts[idx]

        if role == "admin":
            admin_menu()

        elif role == "doctor":
            print(f"\nDoctor {DOCTOR_IDXS.index(idx)+1} login confirmed.")
            input("Press Enter to continue...")
            doctor_menu(idx)

        elif role == "patient":
            # Check registered
            registered = False
            try:
                registered = contract.functions.isPatientRegistered(
                    0  # dummy — events se check karenge
                ).call()
            except:
                pass

            # Events se patient dhundho
            events = contract.events.PatientRegistered.get_logs(from_block=0)
            found = False
            for e in events:
                pid = e['args']['patientId']
                try:
                    addr = contract.functions.getPatientAddress(pid).call()
                    if addr.lower() == address.lower():
                        found = True
                        break
                except:
                    pass

            if not found:
                print(f"\nAccount [{idx}] registered nahi hai.")
                print("Admin se contact karo registration ke liye.")
                input("\nPress Enter to continue...")
                continue

            patient_menu(address)

        else:
            print("Unknown role.")
            input("Press Enter...")

# ── Run ────────────────────────────────────────────────────────────────
main()



DR DIAGNOSIS BLOCKCHAIN SYSTEM
Private Network | Chain ID 1337

Available accounts:
  [0]      Admin
  [1][2][3] Doctors
  [4-19]   Patients

Enter your account number (0-19): 1

Doctor 1 login confirmed.
Press Enter to continue...


DOCTOR 1 PANEL
Address: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
------------------------------
[1] My patients
[2] Pending reviews
[3] Upload diagnosis + Record decision
[4] Exit

Select option: 1

Assigned patients: [101]

Press Enter to continue...


DOCTOR 1 PANEL
Address: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
------------------------------
[1] My patients
[2] Pending reviews
[3] Upload diagnosis + Record decision
[4] Exit

Select option: 2

No pending reviews.

Press Enter to continue...3


DOCTOR 1 PANEL
Address: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
------------------------------
[1] My patients
[2] Pending reviews
[3] Upload diagnosis + Record decision
[4] Exit

Select option: 3
Enter Patient ID: 101
Error: [Errno 2] No such file or d

KeyboardInterrupt: Interrupted by user